# S3 Vectors RAG Walkthrough

End-to-end RAG ingest + retrieval on **Amazon S3 Vectors** with **Bedrock Titan Text Embeddings v2**.

Pipeline:

1. Configure region / bucket / index.
2. Load + section + chunk a small markdown corpus.
3. Ensure vector bucket and index (idempotent).
4. Embed each chunk with Titan v2 and `put_vectors`.
5. Run `query_vectors` with and without metadata filters.
6. Tear everything down.

All cells call the live AWS APIs. Make sure your AWS credentials are configured and your account has access to `s3vectors` and `bedrock-runtime` in the chosen region.

In [ ]:
import os
import sys
from pathlib import Path

# Make the src/ package importable when running the notebook from notebooks/.
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))

from s3vectors_rag.config import Config
from s3vectors_rag.logging_setup import configure
from s3vectors_rag.loaders import load_file, iter_files
from s3vectors_rag.chunking import chunk_text
from s3vectors_rag.embeddings import TitanEmbedder
from s3vectors_rag.store import S3VectorStore, VectorRecord
from s3vectors_rag.ingest import build_chunks, ingest_paths

configure('INFO')

cfg = Config(
    region=os.environ.get('AWS_REGION', 'us-west-2'),
    bucket_name=os.environ.get('S3V_BUCKET', 's3-vectors-rag-nb'),
    index_name=os.environ.get('S3V_INDEX', 'rag-index'),
    embed_dimension=1024,
    chunk_size=300,
    chunk_overlap=40,
)
cfg

## 1. Inspect the corpus

We load every supported file under `samples/` and split each markdown file by `#` / `##` headings. Each section gets a locator like `section=Indexing`, which becomes filterable metadata later.

In [ ]:
samples_dir = ROOT / 'samples'
for fp in iter_files(samples_dir):
    sections = load_file(fp)
    print(f'{fp.name}: {len(sections)} sections')
    for s in sections[:3]:
        preview = s.text[:80].replace(chr(10), ' ')
        print(f'  {s.locator:<30} {preview}...')

## 2. Chunk a single section

The chunker is token-aware (uses `tiktoken` if installed, otherwise whitespace). Each chunk inherits the section's `source` and `locator`, plus its own `chunk_idx`.

In [ ]:
fp = samples_dir / 'aws-rag.md'
first = load_file(fp)[0]
chunks = chunk_text(first.text, chunk_size=120, chunk_overlap=20, base_metadata={'source': fp.name, 'locator': first.locator})
for c in chunks:
    print(c.metadata, '->', c.text[:60].replace(chr(10), ' '), '...')

## 3. Build the full chunk set

`build_chunks` runs the loader + chunker over a list of paths and gives us back uniform records ready for embedding.

In [ ]:
chunks = build_chunks([samples_dir], cfg)
print(f'total chunks: {len(chunks)}')
chunks[0].metadata

## 4. Ensure the vector bucket + index

These calls are idempotent — re-running this notebook won't blow up on `ConflictException`.

In [ ]:
store = S3VectorStore(cfg)
store.ensure()
store.stats()

## 5. Embed + upsert

We use the package's `ingest_paths` helper, which combines load + chunk + embed + put. It returns the number of vectors written.

In [ ]:
written = ingest_paths([samples_dir], cfg, store=store)
print(f'wrote {written} vectors')

## 6. Query — open

The query embedding goes to the index and we ask for the top 3 hits. Scores are derived from cosine distance: `score = 1 - distance` (higher = more similar).

In [ ]:
embedder = TitanEmbedder(cfg)
qvec = embedder.embed_one('How do I store and query vectors in S3?')
for h in store.query(qvec, top_k=3):
    src = h.metadata.get('source')
    loc = h.metadata.get('locator')
    txt = (h.metadata.get('text') or '').replace(chr(10), ' ')[:120]
    print(f'{h.score:.3f}  {src} {loc}\n   {txt}...')

## 7. Query — filtered by metadata

We can scope the search to a single source. The filter shape is the s3vectors `filter` document — equality is `{key: value}`, conjunctions use `$and`.

In [ ]:
qvec = embedder.embed_one('embedding dimensions tradeoff')
for h in store.query(qvec, top_k=3, metadata_filter={'source': 'aws-rag.md'}):
    src = h.metadata.get('source')
    loc = h.metadata.get('locator')
    txt = (h.metadata.get('text') or '').replace(chr(10), ' ')[:120]
    print(f'{h.score:.3f}  {src} {loc}\n   {txt}...')

## 8. Cleanup

S3 Vectors charges for storage — when you're done experimenting, delete the vectors, then the index, then the bucket.

In [ ]:
import boto3
c = boto3.client('s3vectors', region_name=cfg.region)
keys, nt = [], None
while True:
    kw = {'vectorBucketName': cfg.bucket_name, 'indexName': cfg.index_name, 'maxResults': 500}
    if nt: kw['nextToken'] = nt
    r = c.list_vectors(**kw)
    keys.extend([v['key'] for v in r.get('vectors', [])])
    nt = r.get('nextToken')
    if not nt: break
if keys:
    store.delete_by_keys(keys)
store.teardown()
print('cleaned up')